Step - 1 : Import required library

In [1]:
import numpy as np
import pandas as pd

Step - 2 : Load the Dataset

In [2]:
df = pd.read_csv("../dataset/employee_data.csv")

Step - 3 : Explore the Dataset

In [3]:
print(df.head())
print(df.tail())
print(df.sample(3))
print(df.shape)
print(df.columns)
print(df.dtypes)
df.info()
print(df.describe())

   Employee_ID    Name Department  Age  Experience  Salary       City  \
0          101   Rahul         IT   24           2   45000    Chennai   
1          102   Priya         HR   28           5   60000  Bangalore   
2          103    Arun      Sales   30           7   70000    Chennai   
3          104   Kiran         IT   26           3   52000  Hyderabad   
4          105  Ramesh         HR   32           8   75000  Bangalore   

  Joining_Date  
0   2023-01-15  
1   2021-08-10  
2   2019-06-20  
3   2022-03-12  
4   2018-11-25  
   Employee_ID    Name Department  Age  Experience  Salary       City  \
3          104   Kiran         IT   26           3   52000  Hyderabad   
4          105  Ramesh         HR   32           8   75000  Bangalore   
5          106   Divya    Finance   25           2   48000    Chennai   
6          107  Suresh      Sales   29           6   65000  Hyderabad   
7          108   Meena         IT   27           4   58000    Chennai   

  Joining_Date  
3  

Step - 4 : Create a working copy

In [4]:
clean_df = df.copy()

Step - 5 : Standardize Column Names

In [5]:
clean_df.columns = (clean_df.columns.str.strip().str.lower().str.replace(" ","_"))
clean_df.columns

Index(['employee_id', 'name', 'department', 'age', 'experience', 'salary',
       'city', 'joining_date'],
      dtype='str')

Step - 6 : Detect missing values

In [6]:
clean_df.isnull().sum()

employee_id     0
name            0
department      0
age             0
experience      0
salary          0
city            0
joining_date    0
dtype: int64

In [7]:
# missing percentage
missing_percentage = (clean_df.isnull().sum() / len(clean_df)) * 100
missing_percentage

employee_id     0.0
name            0.0
department      0.0
age             0.0
experience      0.0
salary          0.0
city            0.0
joining_date    0.0
dtype: float64

Step - 7 : Remove duplicate records

In [8]:
clean_df = clean_df.drop_duplicates()

In [9]:
# check duplicate count
clean_df.duplicated().sum()

np.int64(0)

Step - 8 : Clean Text Columns

In [10]:
clean_df["name"] = clean_df["name"].str.strip().str.title()
clean_df["department"] = clean_df["department"].str.strip().str.title()
clean_df["city"] = clean_df["city"].str.strip().str.title()
clean_df.head()

,employee_id,name,department,age,experience,salary,city,joining_date
0,101,Rahul,It,24,2,45000,Chennai,2023-01-15
1,102,Priya,Hr,28,5,60000,Bangalore,2021-08-10
2,103,Arun,Sales,30,7,70000,Chennai,2019-06-20
3,104,Kiran,It,26,3,52000,Hyderabad,2022-03-12
4,105,Ramesh,Hr,32,8,75000,Bangalore,2018-11-25


Step - 9 : Replace Inconsistent Values

In [11]:
clean_df["department"] = clean_df["department"].replace({
    "It" : "IT",
    "Hr" : "HR",
    "Information Tech" : "IT",
    "Human Resource" : "HR"
})
clean_df.head()

,employee_id,name,department,age,experience,salary,city,joining_date
0,101,Rahul,IT,24,2,45000,Chennai,2023-01-15
1,102,Priya,HR,28,5,60000,Bangalore,2021-08-10
2,103,Arun,Sales,30,7,70000,Chennai,2019-06-20
3,104,Kiran,IT,26,3,52000,Hyderabad,2022-03-12
4,105,Ramesh,HR,32,8,75000,Bangalore,2018-11-25


Step - 10 : Handle missing values

In [12]:
# fill missing salary with department median
clean_df["salary"] = clean_df["salary"].fillna(clean_df.groupby("department")["salary"].transform("median"))

In [13]:
# fill missing city with mode
clean_df["city"] = clean_df["city"].fillna(clean_df["city"].mode()[0])

Step - 11 : Correct Data Types

In [14]:
clean_df["employee_id"] = clean_df["employee_id"].astype(int)
clean_df["age"] = clean_df["age"].astype(int)
clean_df["experience"] = clean_df["experience"].astype(int)
clean_df["salary"] = clean_df["salary"].astype(int)

Step - 12 : Convert Date column

In [15]:
clean_df["joining_date"] = pd.to_datetime(clean_df["joining_date"],errors="coerce")

Step - 13 : Handle Invalid Records

In [16]:
# remove invalid age
clean_df = clean_df[clean_df["age"].between(18,65)]

In [17]:
# remove negative experience
clean_df = clean_df[clean_df["experience"] >= 0]

In [18]:
# remove invalid salary
clean_df = clean_df[clean_df["salary"] > 0]

Step - 14 : Feature Engineering

In [19]:
# create bonus
# calculate a 10% bonus
clean_df["bonus"] = clean_df["salary"] * 0.10

In [20]:
# create annual salary
clean_df["annual_salary"] = clean_df["salary"] * 12

In [21]:
# create total annual compensation
clean_df["annual_bonus"] = clean_df["bonus"] * 12

clean_df["annual_compensation"] = clean_df["annual_salary"] + clean_df["annual_bonus"]

In [22]:
# create experience level
def classify_experience(years):
    if years<2:
        return "Fresher"
    elif years<5:
        return "Junior"
    elif years<8:
        return "Mid-Level"
    return "Senior"

clean_df["experience_level"] = clean_df["experience"].apply(classify_experience)

In [23]:
# create salary level
clean_df["salary_level"] = pd.cut(
    clean_df["salary"],
    bins = [0,50000,65000,float("inf")],
    labels = ["Low","Medium","High"],
    include_lowest=True
)

In [24]:
# create joining year
clean_df["joining_year"] = clean_df["joining_date"].dt.year

In [25]:
# create joining month
clean_df["joining_month"] = clean_df["joining_date"].dt.month_name()

In [26]:
# calculate employment duration
today = pd.Timestamp.today()
clean_df["service_days"] = (today - clean_df["joining_date"]).dt.days

In [27]:
# approximate years
clean_df["service_years"] = (clean_df["service_days"] / 365.25).round(1)

In [28]:
# create salary rank
clean_df["salary_rank"] = (
    clean_df["salary"]
    .rank(
        ascending=False,
        method="dense"
    )
    .astype(int)
)

In [29]:
clean_df.head()

,employee_id,name,department,age,experience,salary,city,joining_date,bonus,annual_salary,annual_bonus,annual_compensation,experience_level,salary_level,joining_year,joining_month,service_days,service_years,salary_rank
0,101,Rahul,IT,24,2,45000,Chennai,2023-01-15,4500.0,540000,54000.0,594000.0,Junior,Low,2023,January,1353,3.7,8
1,102,Priya,HR,28,5,60000,Bangalore,2021-08-10,6000.0,720000,72000.0,792000.0,Mid-Level,Medium,2021,August,1876,5.1,4
2,103,Arun,Sales,30,7,70000,Chennai,2019-06-20,7000.0,840000,84000.0,924000.0,Mid-Level,High,2019,June,2658,7.3,2
3,104,Kiran,IT,26,3,52000,Hyderabad,2022-03-12,5200.0,624000,62400.0,686400.0,Junior,Medium,2022,March,1662,4.6,6
4,105,Ramesh,HR,32,8,75000,Bangalore,2018-11-25,7500.0,900000,90000.0,990000.0,Senior,High,2018,November,2865,7.8,1


Step - 15 : Basic Data Analysis

In [30]:
# average salary
clean_df["salary"].mean()

np.float64(59125.0)

In [31]:
# highest salary
clean_df["salary"].max()

np.int64(75000)

In [32]:
# lowest salary
clean_df["salary"].min()

np.int64(45000)

In [33]:
# total salary expense
clean_df["salary"].sum()

np.int64(473000)

In [34]:
# employee count
clean_df["employee_id"].nunique()

8

In [35]:
# highest paid employee
clean_df.loc[clean_df["salary"].idxmax()]

employee_id                            105
name                                Ramesh
department                              HR
age                                     32
experience                               8
salary                               75000
city                             Bangalore
joining_date           2018-11-25 00:00:00
bonus                               7500.0
annual_salary                       900000
annual_bonus                       90000.0
annual_compensation               990000.0
experience_level                    Senior
salary_level                          High
joining_year                          2018
joining_month                     November
service_days                          2865
service_years                          7.8
salary_rank                              1
Name: 4, dtype: object

Step - 16 : Department-Wise Analysis

In [36]:
department_report = clean_df.groupby("department").agg(
    employee_count=("employee_id","count"),
    total_salary=("salary","sum"),
    average_salary=("salary","mean"),
    highest_salary=("salary","max"),
    average_experience=("experience","mean")
)

In [37]:
# sort by total salary
department_report = department_report.sort_values("total_salary",ascending=False)
department_report

,employee_count,total_salary,average_salary,highest_salary,average_experience
department,,,,,
IT,3,155000,51666.666667,58000,3.0
HR,2,135000,67500.000000,75000,6.5
Sales,2,135000,67500.000000,70000,6.5
Finance,1,48000,48000.000000,48000,2.0


Step - 17 : City-Wise Analysis

In [38]:
city_report = clean_df.groupby("city").agg(
    employee_count=("employee_id","count"),
    total_salary=("salary","sum"),
    average_salary=("salary","mean"),
)
city_report

,employee_count,total_salary,average_salary
city,,,
Bangalore,2,135000,67500.0
Chennai,4,221000,55250.0
Hyderabad,2,117000,58500.0


Step - 18 : Experience_Level Analysis

In [39]:
experience_report = clean_df.groupby("experience_level",observed=True).agg(
    employee_count=("employee_id","count"),
    average_salary=("salary","mean"),
)
experience_report

,employee_count,average_salary
experience_level,,
Junior,4,50750.0
Mid-Level,3,65000.0
Senior,1,75000.0


Step - 19 : Employee Distribution

In [40]:
# department wise employee count
clean_df["department"].value_counts()

department
IT         3
HR         2
Sales      2
Finance    1
Name: count, dtype: int64

In [41]:
# city wise employee count
clean_df["city"].value_counts()

city
Chennai      4
Bangalore    2
Hyderabad    2
Name: count, dtype: int64

In [42]:
# experience level count
clean_df["experience_level"].value_counts()

experience_level
Junior       4
Mid-Level    3
Senior       1
Name: count, dtype: int64

Step - 20 : Pivot Reports

In [43]:
# department wise salary pivot
department_salary_pivot = pd.pivot_table(
    clean_df,
    values="salary",
    index="department",
    aggfunc=["sum","mean","max","min"]
)
department_salary_pivot

,sum,mean,max,min
,salary,salary,salary,salary
department,,,,
Finance,48000,48000.000000,48000,48000
HR,135000,67500.000000,75000,60000
IT,155000,51666.666667,58000,45000
Sales,135000,67500.000000,70000,65000


In [44]:
# department and city salary report
department_city_pivot = pd.pivot_table(
    clean_df,
    values="salary",
    index="department",
    columns="city",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total"
)
department_city_pivot

city,Bangalore,Chennai,Hyderabad,Total
department,,,,
Finance,0,48000,0,48000
HR,135000,0,0,135000
IT,0,103000,52000,155000
Sales,0,70000,65000,135000
Total,135000,221000,117000,473000


In [45]:
# department and city employee count
employee_distribution = pd.crosstab(
    clean_df["department"],
    clean_df["city"],
    margins=True,
    margins_name="Total"
)
employee_distribution

city,Bangalore,Chennai,Hyderabad,Total
department,,,,
Finance,0,1,0,1
HR,2,0,0,2
IT,0,2,1,3
Sales,0,1,1,2
Total,2,4,2,8


Step - 21 : Top Five Employees by Salary

In [46]:
top_five_employees = clean_df.nlargest(5,"salary")[["employee_id","name","department","salary","salary_rank"]]
top_five_employees

,employee_id,name,department,salary,salary_rank
4,105,Ramesh,HR,75000,1
2,103,Arun,Sales,70000,2
6,107,Suresh,Sales,65000,3
1,102,Priya,HR,60000,4
7,108,Meena,IT,58000,5


Step - 22 : Employees Joined after a specific date

In [47]:
recent_employees = clean_df[clean_df["joining_date"] > "2022-01-01"]
recent_employees

,employee_id,name,department,age,experience,salary,city,joining_date,bonus,annual_salary,annual_bonus,annual_compensation,experience_level,salary_level,joining_year,joining_month,service_days,service_years,salary_rank
0,101,Rahul,IT,24,2,45000,Chennai,2023-01-15,4500.0,540000,54000.0,594000.0,Junior,Low,2023,January,1353,3.7,8
3,104,Kiran,IT,26,3,52000,Hyderabad,2022-03-12,5200.0,624000,62400.0,686400.0,Junior,Medium,2022,March,1662,4.6,6
5,106,Divya,Finance,25,2,48000,Chennai,2023-07-01,4800.0,576000,57600.0,633600.0,Junior,Low,2023,July,1186,3.2,7


Step - 23 : Generate Business Insight

The project should not stop with calculations. Students must convert results into business insights

Sample Insights

The department with the highest total salary expense requires the largest payroll budget.

The city with the highest employee count may require additional office resources.

Step - 24 : Export the clean Dataset

In [48]:
# CSV Export
clean_df.to_csv("../reports/employee_report.csv",index=False)

In [49]:
# Excel Export
clean_df.to_excel("../reports/employee_report.xlsx",index=False)

Step - 25 : Export Multiple reports to Excel

In [50]:
with pd.ExcelWriter("../reports/employee_analysis_report.xlsx") as writer:
    clean_df.to_excel(writer,sheet_name="Clean Employee Data",index=False),
    department_report.to_excel(writer,sheet_name="Department Analaysis"),
    city_report.to_excel(writer,sheet_name="City Analysis"),
    experience_report.to_excel(writer,sheet_name="Experience Analysis"),
    department_city_pivot.to_excel(writer,sheet_name="Salary Pivot"),
    employee_distribution.to_excel(writer,sheet_name="Employee Distribution"),
    top_five_employees.to_excel(writer,sheet_name="Top Employees",index=False)

